# Fabric Workspace Refresh Orchestrator (DAG + collision-aware)

This notebook is a stronger replacement for the earlier phase-based notebook.

## What it does
- inventories the workspace
- builds a dependency DAG automatically where public APIs expose lineage
- prevents refresh collisions with source-based concurrency guards
- uses topological scheduling instead of one-big-phase-at-a-time
- refreshes more aggressively but safely
- logs **SUCCESS / FAILED / SKIPPED / BLOCKED**
- exports results to CSV

## Important reality check
- **Reports do not refresh**. Their underlying **semantic models** and **dataflows** refresh.
- **Paginated reports (RDL)** are inventoried for visibility, but they are not submitted as refresh jobs here.
- **Automatic dependency detection is strongest for Dataflow Gen1 and semantic models** because Power BI exposes official upstream-dataflow endpoints for those.
- **Dataflow Gen2 cross-item lineage is still weaker in public APIs**, so this notebook combines: 
  1. official lineage where available,
  2. datasource-key collision protection,
  3. type precedence fallback,
  4. optional manual overrides for edge cases.

In [ ]:
from notebookutils import mssparkutils
import requests, json, time, os, math, random, hashlib
from datetime import datetime, timezone
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, FIRST_COMPLETED, wait

# ---------------------------
# REQUIRED CONFIG
# ---------------------------
WORKSPACE_ID = "8fac9a67-4af2-4305-8b44-d63f5b5a6e91"

# ---------------------------
# SCHEDULER TUNING
# ---------------------------
MAX_GLOBAL_WORKERS = 8
MAX_PER_SOURCE_DOMAIN = 1        # key guard against source collisions
MAX_GEN2_WORKERS = 4
MAX_GEN1_WORKERS = 3
MAX_DATASET_WORKERS = 5

POLL_SECONDS = 20
API_TIMEOUT_SECONDS = 90
GEN2_TIMEOUT_MIN = 120
GEN1_TIMEOUT_MIN = 120
DATASET_TIMEOUT_MIN = 240

# Retry settings
MAX_TRIGGER_RETRIES = 3
BACKOFF_SECONDS = [8, 20, 45]
JITTER_MAX = 4

# Enhanced refresh for semantic models.
# On shared capacity only notifyOption is supported. On Premium/Fabric capacity,
# enhanced refresh can materially improve throughput for large models.
DATASET_REFRESH_PAYLOAD = {
    # Leave only notifyOption if you want strict compatibility with shared capacity.
    "type": "automatic",
    "commitMode": "transactional",
    "retryCount": 0,
    "maxParallelism": 4,
    "applyRefreshPolicy": True
}

# Optional manual dependency overrides
# Example: {("DataflowGen2", "child_id"): [("DataflowGen2", "parent_id"), ("DataflowGen1", "parent2_id")]} 
MANUAL_UPSTREAM_OVERRIDES = {}

# Optional exclusions
EXCLUDE_ITEM_IDS = set()
EXCLUDE_DATASET_NAMES = set()
EXCLUDE_DATAFLOW_NAMES = set()

# Output folder
OUTPUT_DIR = "/lakehouse/default/Files/fabric_refresh_orchestrator"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FABRIC_BASE = "https://api.fabric.microsoft.com/v1"
PBI_BASE = "https://api.powerbi.com/v1.0/myorg"

print("Config loaded.")

In [ ]:
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def now_compact():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def jitter_sleep(base_seconds):
    time.sleep(base_seconds + random.uniform(0, JITTER_MAX))

def sha1_text(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def normalize_text(x):
    return str(x or "").strip().lower()

def get_fabric_headers():
    token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/.default")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def get_pbi_headers():
    token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api/.default")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def http_get(url, headers, timeout=API_TIMEOUT_SECONDS):
    r = requests.get(url, headers=headers, timeout=timeout)
    if not r.ok:
        raise RuntimeError(f"GET {url} failed: {r.status_code} {r.text}")
    if not r.text:
        return {}
    return r.json()


def http_post(url, headers, payload=None, timeout=API_TIMEOUT_SECONDS):
    r = requests.post(url, headers=headers, json=payload, timeout=timeout)
    if not r.ok:
        raise RuntimeError(f"POST {url} failed: {r.status_code} {r.text}")
    return r


def fabric_get(url, timeout=API_TIMEOUT_SECONDS):
    return http_get(url, get_fabric_headers(), timeout)


def fabric_post(url, payload=None, timeout=API_TIMEOUT_SECONDS):
    return http_post(url, get_fabric_headers(), payload, timeout)


def pbi_get(url, timeout=API_TIMEOUT_SECONDS):
    return http_get(url, get_pbi_headers(), timeout)


def pbi_post(url, payload=None, timeout=API_TIMEOUT_SECONDS):
    return http_post(url, get_pbi_headers(), payload, timeout)


def paginate_value(url, api="fabric"):
    rows = []
    next_url = url
    while next_url:
        obj = fabric_get(next_url) if api == "fabric" else pbi_get(next_url)
        rows.extend(obj.get("value", []))
        next_url = obj.get("continuationUri") or obj.get("@odata.nextLink") or obj.get("nextLink")
    return rows


def safe_name(o, *keys, default=None):
    cur = o
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return cur if cur is not None else default

## Inventory the workspace

In [ ]:
workspace = pbi_get(f"{PBI_BASE}/groups/{WORKSPACE_ID}")
workspace_name = workspace.get("name", WORKSPACE_ID)

reports = pbi_get(f"{PBI_BASE}/groups/{WORKSPACE_ID}/reports").get("value", [])
datasets = pbi_get(f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets").get("value", [])
dataflows_gen1 = pbi_get(f"{PBI_BASE}/groups/{WORKSPACE_ID}/dataflows").get("value", [])
all_fabric_items = paginate_value(f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/items", api="fabric")

# Fabric returns type='Dataflow' for Gen2 items.
dataflows_gen2 = [x for x in all_fabric_items if normalize_text(x.get("type")) == "dataflow"]
paginated_reports = [r for r in reports if normalize_text(r.get("reportType")) == "paginatedreport"]
interactive_reports = [r for r in reports if normalize_text(r.get("reportType")) != "paginatedreport"]

print(f"Workspace: {workspace_name}")
print(f"Reports: {len(reports)} (interactive={len(interactive_reports)}, paginated={len(paginated_reports)})")
print(f"Semantic models: {len(datasets)}")
print(f"Dataflow Gen1: {len(dataflows_gen1)}")
print(f"Dataflow Gen2: {len(dataflows_gen2)}")
print(f"All Fabric items: {len(all_fabric_items)}")

## Build a canonical node catalog

In [ ]:
NODE_TYPE_ORDER = {
    "DataflowGen2": 1,
    "DataflowGen1": 2,
    "SemanticModel": 3,
}

nodes = {}

for df in dataflows_gen2:
    item_id = df["id"]
    name = df.get("displayName") or item_id
    if item_id in EXCLUDE_ITEM_IDS or name in EXCLUDE_DATAFLOW_NAMES:
        continue
    nodes[("DataflowGen2", item_id)] = {
        "kind": "DataflowGen2",
        "id": item_id,
        "name": name,
        "raw": df,
        "upstream": set(),
        "downstream": set(),
        "source_keys": set(),
    }

for df in dataflows_gen1:
    item_id = df["objectId"]
    name = df.get("name") or item_id
    if item_id in EXCLUDE_ITEM_IDS or name in EXCLUDE_DATAFLOW_NAMES:
        continue
    nodes[("DataflowGen1", item_id)] = {
        "kind": "DataflowGen1",
        "id": item_id,
        "name": name,
        "raw": df,
        "upstream": set(),
        "downstream": set(),
        "source_keys": set(),
    }

for ds in datasets:
    item_id = ds["id"]
    name = ds.get("name") or item_id
    if item_id in EXCLUDE_ITEM_IDS or name in EXCLUDE_DATASET_NAMES:
        continue
    nodes[("SemanticModel", item_id)] = {
        "kind": "SemanticModel",
        "id": item_id,
        "name": name,
        "raw": ds,
        "upstream": set(),
        "downstream": set(),
        "source_keys": set(),
    }

print(f"Refreshable nodes loaded: {len(nodes)}")

## Official lineage discovery

This uses official Power BI lineage endpoints where available:
- Dataflow Gen1 → upstream dataflows
- Semantic models → upstream dataflows

For Gen2, public lineage coverage is still not as complete for generic cross-item orchestration, so the notebook uses fallback ordering and collision domains for safety.

In [ ]:
def add_edge(parent_key, child_key):
    if parent_key == child_key:
        return
    if parent_key not in nodes or child_key not in nodes:
        return
    nodes[child_key]["upstream"].add(parent_key)
    nodes[parent_key]["downstream"].add(child_key)


def load_gen1_lineage():
    for key, node in list(nodes.items()):
        if node["kind"] != "DataflowGen1":
            continue
        dfid = node["id"]
        try:
            url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/dataflows/{dfid}/upstreamDataflows"
            ups = pbi_get(url).get("value", [])
            for up in ups:
                upid = up.get("targetDataflowId") or up.get("objectId") or up.get("dataflowObjectId") or up.get("id")
                if not upid:
                    continue
                # Upstream returned by this endpoint is Gen1 lineage.
                add_edge(("DataflowGen1", upid), key)
        except Exception as e:
            print(f"WARN gen1 lineage {node['name']}: {e}")


def load_dataset_lineage():
    try:
        url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/upstreamDataflows"
        links = pbi_get(url).get("value", [])
    except Exception as e:
        print(f"WARN dataset lineage failed: {e}")
        return

    # The response shape can vary. We normalize conservatively.
    for row in links:
        dsid = row.get("datasetObjectId") or row.get("datasetId") or row.get("id")
        if not dsid:
            continue
        child_key = ("SemanticModel", dsid)
        if child_key not in nodes:
            continue
        upstreams = row.get("upstreamDataflows") or row.get("dataflows") or row.get("value") or []
        for up in upstreams:
            upid = up.get("objectId") or up.get("targetDataflowId") or up.get("id")
            if not upid:
                continue
            # Dataset upstream endpoint is a Power BI dataflow lineage API, so treat as Gen1.
            add_edge(("DataflowGen1", upid), child_key)

load_gen1_lineage()
load_dataset_lineage()

# Manual override hook
for child_key, parents in MANUAL_UPSTREAM_OVERRIDES.items():
    for p in parents:
        add_edge(p, child_key)

edge_count = sum(len(n["upstream"]) for n in nodes.values())
print(f"Official/manual lineage edges loaded: {edge_count}")

## Datasource collision domains

Even when lineage is incomplete, two items that hit the same source system should not be blasted at once.
This notebook creates a normalized **source key** for each node and allows only a limited number of concurrent refreshes per source domain.

In [ ]:
def canonical_datasource_key(obj):
    kind = normalize_text(obj.get("datasourceType") or obj.get("connectionType") or obj.get("type") or obj.get("kind"))
    cd = obj.get("connectionDetails") or {}
    parts = [
        kind,
        normalize_text(cd.get("server") or cd.get("path") or cd.get("url") or cd.get("account") or cd.get("domain") or cd.get("loginServer")),
        normalize_text(cd.get("database") or cd.get("siteUrl") or cd.get("workspaceId") or cd.get("warehouse") or cd.get("lakehouse") or cd.get("folderPath")),
    ]
    parts = [p for p in parts if p]
    if not parts:
        return None
    return "|".join(parts)


def load_sources_for_gen1():
    for key, node in list(nodes.items()):
        if node["kind"] != "DataflowGen1":
            continue
        try:
            url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/dataflows/{node['id']}/datasources"
            dsources = pbi_get(url).get("value", [])
            for s in dsources:
                k = canonical_datasource_key(s)
                if k:
                    node["source_keys"].add(k)
        except Exception as e:
            print(f"WARN gen1 datasources {node['name']}: {e}")


def load_sources_for_datasets():
    for key, node in list(nodes.items()):
        if node["kind"] != "SemanticModel":
            continue
        try:
            url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{node['id']}/datasources"
            dsources = pbi_get(url).get("value", [])
            for s in dsources:
                k = canonical_datasource_key(s)
                if k:
                    node["source_keys"].add(k)
        except Exception as e:
            print(f"WARN dataset datasources {node['name']}: {e}")


def load_sources_for_gen2():
    # Generic datasource discovery for Gen2 is not consistently exposed in the same way,
    # so we assign a stable per-item domain when no explicit source keys are available.
    for key, node in list(nodes.items()):
        if node["kind"] != "DataflowGen2":
            continue
        if not node["source_keys"]:
            node["source_keys"].add(f"gen2-item|{node['id']}")

load_sources_for_gen1()
load_sources_for_datasets()
load_sources_for_gen2()

source_key_count = sum(len(n["source_keys"]) for n in nodes.values())
print(f"Source keys loaded: {source_key_count}")

## Fallback ordering for nodes without explicit lineage

If an item has no lineage edges, the scheduler still respects a conservative type precedence:
Dataflow Gen2 → Dataflow Gen1 → Semantic model.

In [ ]:
def can_start_by_type_precedence(node, finished_kinds):
    kind = node["kind"]
    if node["upstream"]:
        return True
    # Conservative fallback: don't start later layers before earlier layers have at least drained.
    if kind == "DataflowGen2":
        return True
    if kind == "DataflowGen1":
        return "DataflowGen2" in finished_kinds or not any(n["kind"] == "DataflowGen2" for n in nodes.values())
    if kind == "SemanticModel":
        need_gen2 = any(n["kind"] == "DataflowGen2" for n in nodes.values())
        need_gen1 = any(n["kind"] == "DataflowGen1" for n in nodes.values())
        return ((not need_gen2) or ("DataflowGen2" in finished_kinds)) and ((not need_gen1) or ("DataflowGen1" in finished_kinds))
    return True

## Trigger + wait helpers

In [ ]:
def get_gen2_jobs(item_id):
    url = f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/items/{item_id}/jobs/instances"
    return fabric_get(url).get("value", [])


def get_gen1_transactions(item_id):
    url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/dataflows/{item_id}/transactions"
    return pbi_get(url).get("value", [])


def get_dataset_refreshes(item_id, top=10):
    url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{item_id}/refreshes?$top={top}"
    return pbi_get(url).get("value", [])


def latest_active_status(kind, item_id):
    if kind == "DataflowGen2":
        rows = get_gen2_jobs(item_id)
        active = [r for r in rows if normalize_text(r.get("status")) in {"notstarted", "queued", "inprogress", "running"}]
        active.sort(key=lambda x: x.get("startTime", ""), reverse=True)
        return active[0] if active else None
    if kind == "DataflowGen1":
        rows = get_gen1_transactions(item_id)
        active = [r for r in rows if normalize_text(r.get("status")) in {"notstarted", "queued", "inprogress", "running", "unknown"}]
        active.sort(key=lambda x: x.get("startTime", ""), reverse=True)
        return active[0] if active else None
    if kind == "SemanticModel":
        rows = get_dataset_refreshes(item_id, top=10)
        active = [r for r in rows if normalize_text(r.get("status")) in {"notstarted", "queued", "inprogress", "running", "unknown"}]
        active.sort(key=lambda x: x.get("startTime", ""), reverse=True)
        return active[0] if active else None
    return None


def trigger_gen2(item_id):
    url = f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/dataflows/{item_id}/jobs/Execute/instances"
    resp = fabric_post(url, payload={})
    return {"statusCode": resp.status_code, "location": resp.headers.get("Location")}


def trigger_gen1(item_id):
    url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/dataflows/{item_id}/refreshes"
    resp = pbi_post(url, payload={"notifyOption": "NoNotification"})
    return {"statusCode": resp.status_code}


def trigger_dataset(item_id):
    url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{item_id}/refreshes"
    payload = dict(DATASET_REFRESH_PAYLOAD)
    try:
        resp = pbi_post(url, payload=payload)
        return {"statusCode": resp.status_code, "mode": "enhanced"}
    except Exception as e:
        # Fallback for shared capacity / unsupported enhanced options.
        payload = {"notifyOption": "NoNotification"}
        resp = pbi_post(url, payload=payload)
        return {"statusCode": resp.status_code, "mode": "standard"}


def wait_for_completion(kind, item_id, timeout_minutes):
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        if kind == "DataflowGen2":
            rows = get_gen2_jobs(item_id)
        elif kind == "DataflowGen1":
            rows = get_gen1_transactions(item_id)
        else:
            rows = get_dataset_refreshes(item_id, top=10)
        rows = sorted(rows, key=lambda x: x.get("startTime", ""), reverse=True)
        if rows:
            latest = rows[0]
            status = normalize_text(latest.get("status"))
            if status in {"success", "succeeded", "completed"}:
                return "SUCCESS", latest
            if status in {"failed", "cancelled", "canceled"}:
                return "FAILED", latest
        time.sleep(POLL_SECONDS)
    return "FAILED", {"status": "TimedOut"}


def trigger_with_retry(node):
    kind = node["kind"]
    item_id = node["id"]
    name = node["name"]

    active = latest_active_status(kind, item_id)
    if active:
        return "SKIPPED", f"Already running ({active.get('status')})", active

    last_error = None
    for attempt in range(1, MAX_TRIGGER_RETRIES + 1):
        try:
            if kind == "DataflowGen2":
                info = trigger_gen2(item_id)
                final, raw = wait_for_completion(kind, item_id, GEN2_TIMEOUT_MIN)
            elif kind == "DataflowGen1":
                info = trigger_gen1(item_id)
                final, raw = wait_for_completion(kind, item_id, GEN1_TIMEOUT_MIN)
            else:
                info = trigger_dataset(item_id)
                final, raw = wait_for_completion(kind, item_id, DATASET_TIMEOUT_MIN)
            return final, f"Triggered {info}; final={raw.get('status')}", raw
        except Exception as e:
            last_error = str(e)
            if attempt < MAX_TRIGGER_RETRIES:
                jitter_sleep(BACKOFF_SECONDS[min(attempt - 1, len(BACKOFF_SECONDS) - 1)])
            else:
                break
    return "FAILED", last_error or "Unknown trigger failure", None

## DAG-aware collision-safe scheduler

In [ ]:
results = []

def log_result(node, status, detail, started_at=None, ended_at=None, extra=None):
    row = {
        "timestampUtc": utc_now_iso(),
        "kind": node["kind"],
        "itemId": node["id"],
        "itemName": node["name"],
        "status": status,
        "detail": detail,
        "startedAtUtc": started_at,
        "endedAtUtc": ended_at,
        "sourceKeys": sorted(node.get("source_keys", set())),
        "upstreamCount": len(node.get("upstream", set())),
        "downstreamCount": len(node.get("downstream", set())),
    }
    if extra:
        row.update(extra)
    results.append(row)


def max_workers_for_kind(kind):
    if kind == "DataflowGen2":
        return MAX_GEN2_WORKERS
    if kind == "DataflowGen1":
        return MAX_GEN1_WORKERS
    if kind == "SemanticModel":
        return MAX_DATASET_WORKERS
    return 1


def kind_running_count(inflight, kind):
    return sum(1 for _, n in inflight.values() if n["kind"] == kind)


def has_source_capacity(node, source_usage):
    for sk in node["source_keys"]:
        if source_usage.get(sk, 0) >= MAX_PER_SOURCE_DOMAIN:
            return False
    return True


def claim_sources(node, source_usage):
    for sk in node["source_keys"]:
        source_usage[sk] = source_usage.get(sk, 0) + 1


def release_sources(node, source_usage):
    for sk in node["source_keys"]:
        source_usage[sk] = max(0, source_usage.get(sk, 0) - 1)


def upstream_finished(node_key, node_statuses):
    node = nodes[node_key]
    for p in node["upstream"]:
        if node_statuses.get(p) != "SUCCESS":
            return False
    return True


def upstream_blocked(node_key, node_statuses):
    node = nodes[node_key]
    return any(node_statuses.get(p) in {"FAILED", "BLOCKED"} for p in node["upstream"])


def worker_run(node):
    started_at = utc_now_iso()
    status, detail, raw = trigger_with_retry(node)
    ended_at = utc_now_iso()
    return {
        "node": node,
        "status": status,
        "detail": detail,
        "startedAtUtc": started_at,
        "endedAtUtc": ended_at,
        "raw": raw,
    }


def run_scheduler():
    node_keys = list(nodes.keys())
    node_statuses = {}
    inflight = {}
    source_usage = {}
    finished_kinds = set()

    # Pre-log paginated reports as skipped for visibility.
    for r in paginated_reports:
        results.append({
            "timestampUtc": utc_now_iso(),
            "kind": "PaginatedReport",
            "itemId": r["id"],
            "itemName": r.get("name") or r["id"],
            "status": "SKIPPED",
            "detail": "RDL report inventoried only. Refresh underlying semantic model or datasource instead.",
            "startedAtUtc": None,
            "endedAtUtc": None,
            "sourceKeys": [],
            "upstreamCount": 0,
            "downstreamCount": 0,
        })

    with ThreadPoolExecutor(max_workers=MAX_GLOBAL_WORKERS) as ex:
        while len(node_statuses) < len(node_keys):
            progress = False

            # Block nodes whose prerequisites failed.
            for nk in node_keys:
                if nk in node_statuses:
                    continue
                if upstream_blocked(nk, node_statuses):
                    node_statuses[nk] = "BLOCKED"
                    log_result(nodes[nk], "BLOCKED", "An upstream dependency failed or was blocked.")
                    progress = True

            # Start ready nodes.
            for nk in sorted(node_keys, key=lambda x: (NODE_TYPE_ORDER.get(x[0], 99), nodes[x]["name"])):
                if nk in node_statuses:
                    continue
                if any(nk == v[1] for v in inflight.values()):
                    continue

                node = nodes[nk]

                # True DAG gating when upstream edges exist.
                if node["upstream"] and not upstream_finished(nk, node_statuses):
                    continue

                # Fallback type precedence when lineage is missing.
                if not can_start_by_type_precedence(node, finished_kinds):
                    continue

                # Enforce per-kind and per-source concurrency.
                if kind_running_count(inflight, node["kind"]) >= max_workers_for_kind(node["kind"]):
                    continue
                if not has_source_capacity(node, source_usage):
                    continue

                claim_sources(node, source_usage)
                fut = ex.submit(worker_run, node)
                inflight[fut] = (utc_now_iso(), nk)
                progress = True

                if len(inflight) >= MAX_GLOBAL_WORKERS:
                    break

            # If nothing started, wait for one active worker.
            if inflight:
                done, _ = wait(list(inflight.keys()), timeout=POLL_SECONDS, return_when=FIRST_COMPLETED)
                for fut in done:
                    started_marker, nk = inflight.pop(fut)
                    node = nodes[nk]
                    release_sources(node, source_usage)
                    out = fut.result()
                    node_statuses[nk] = out["status"]
                    if all(node_statuses.get(k) in {"SUCCESS", "FAILED", "BLOCKED", "SKIPPED"} for k, n in nodes.items() if n["kind"] == node["kind"]):
                        finished_kinds.add(node["kind"])
                    log_result(node, out["status"], out["detail"], out["startedAtUtc"], out["endedAtUtc"])
                    print(f"{out['status']}: {node['kind']} | {node['name']} | {out['detail']}")
                    progress = True
            elif not progress:
                # Deadlock safety.
                unresolved = [k for k in node_keys if k not in node_statuses]
                for nk in unresolved:
                    node_statuses[nk] = "BLOCKED"
                    log_result(nodes[nk], "BLOCKED", "Scheduler deadlock prevention: unresolved prerequisites or capacity guard.")
                break

    return node_statuses

node_statuses = run_scheduler()
print("Scheduler finished.")

## Final summary + export

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values(["status", "kind", "itemName"], ascending=[True, True, True]).reset_index(drop=True)

summary = results_df.groupby(["kind", "status"]).size().reset_index(name="count") if not results_df.empty else pd.DataFrame(columns=["kind", "status", "count"])
print("
Summary by kind/status")
display(summary)

out_csv = os.path.join(OUTPUT_DIR, f"fabric_refresh_results_{now_compact()}.csv")
results_df.to_csv(out_csv, index=False)
print(f"CSV exported to: {out_csv}")

## How this is faster than the old notebook

The old notebook waits for whole phases to finish. This one does not.

Example:
- when a Dataflow Gen1 item finishes, any dependent semantic model can become eligible immediately
- unrelated items continue in parallel
- items hitting the same source system are throttled to prevent self-inflicted contention

That combination is typically where the time savings come from.

## Practical tuning notes

Start with these values and increase gradually:
- `MAX_GLOBAL_WORKERS = 8`
- `MAX_GEN2_WORKERS = 4`
- `MAX_GEN1_WORKERS = 3`
- `MAX_DATASET_WORKERS = 5`
- `MAX_PER_SOURCE_DOMAIN = 1`

If your capacity and source systems are strong, raise one number at a time.
If you see throttling, timeouts, or gateway pressure, lower `MAX_GLOBAL_WORKERS` first.